# Demo: `backtest()` method

This notebook demonstrates the `ForecastingAssistant.backtest()` method for every supported forecaster type.

The backtesting workflow:
1. Profiles the data
2. Generates a plan
3. Creates a cross-validation strategy (`TimeSeriesFold`)
4. Generates and executes backtesting code via `exec()`
5. Returns a `BacktestResult` with metrics, predictions, and the exact code that was run

**Key guarantee:** `result.code` is the exact script that produced `result.metrics` and `result.predictions`.

We show two scenarios for each forecaster:
- **Without exogenous variables**
- **With exogenous variables**

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import warnings

import numpy as np
import pandas as pd
from skforecast.model_selection import TimeSeriesFold

from skforecast_ai import ForecastingAssistant

warnings.filterwarnings("ignore")
assistant = ForecastingAssistant()

/opt/homebrew/Caskroom/miniconda/base/envs/skforecast_ai_py13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Synthetic Data

We create datasets for each forecaster type, with and without exogenous variables.

In [3]:
rng = np.random.default_rng(42)
n = 365
dates = pd.date_range("2022-01-01", periods=n, freq="D")

# Single series target
target = 100 + np.cumsum(rng.normal(0, 1, n)) + 5 * np.sin(2 * np.pi * np.arange(n) / 7)

# Exogenous variables
promo = rng.normal(50, 10, n)
temperature = 15 + 10 * np.sin(2 * np.pi * np.arange(n) / 365) + rng.normal(0, 2, n)

# Single series without exog
df_single = pd.DataFrame({
    "date": dates,
    "sales": target,
})

# Single series with exog
df_single_exog = pd.DataFrame({
    "date": dates,
    "sales": target,
    "promo": promo,
    "temperature": temperature,
})

# Multi-series (long format) without exog
n_multi = 200
dates_multi = pd.date_range("2022-01-01", periods=n_multi, freq="D")
df_multi = pd.DataFrame({
    "date": np.tile(dates_multi, 3),
    "series_id": ["store_A"] * n_multi + ["store_B"] * n_multi + ["store_C"] * n_multi,
    "revenue": np.concatenate([
        100 + np.cumsum(rng.normal(0, 1, n_multi)),
        200 + np.cumsum(rng.normal(0, 1.5, n_multi)),
        150 + np.cumsum(rng.normal(0, 0.8, n_multi)),
    ]),
})

# Multi-series (long format) with exog
df_multi_exog = df_multi.copy()
df_multi_exog["promo"] = rng.normal(50, 10, n_multi * 3)

# Wide format for multivariate without exog
df_wide = pd.DataFrame({
    "date": dates[:n_multi],
    "temperature": 20 + np.cumsum(rng.normal(0, 0.5, n_multi)),
    "humidity": 60 + np.cumsum(rng.normal(0, 0.3, n_multi)),
    "pressure": 1013 + np.cumsum(rng.normal(0, 0.2, n_multi)),
})

# Wide format for multivariate with exog
df_wide_exog = pd.DataFrame({
    "date": dates[:n_multi],
    "temperature": 20 + np.cumsum(rng.normal(0, 0.5, n_multi)),
    "humidity": 60 + np.cumsum(rng.normal(0, 0.3, n_multi)),
    "pressure": 1013 + np.cumsum(rng.normal(0, 0.2, n_multi)),
    "promo": rng.normal(50, 10, n_multi),
})

print(f"df_single:      {df_single.shape}")
print(f"df_single_exog: {df_single_exog.shape}")
print(f"df_multi:       {df_multi.shape}")
print(f"df_multi_exog:  {df_multi_exog.shape}")
print(f"df_wide:        {df_wide.shape}")
print(f"df_wide_exog:   {df_wide_exog.shape}")

df_single:      (365, 2)
df_single_exog: (365, 4)
df_multi:       (600, 3)
df_multi_exog:  (600, 4)
df_wide:        (200, 4)
df_wide_exog:   (200, 5)


---
## 1. ForecasterRecursive (single series)

Recursive multi-step prediction using a single global model.

### Without exogenous variables

In [4]:
# Step-by-step: profile → plan → create_cv → backtest
profile_recursive = assistant.profile(
    data        = df_single,
    target      = "sales",
    date_column = "date",
)

plan_recursive = assistant.plan(
    profile    = profile_recursive,
    steps      = 14,
    forecaster = "ForecasterRecursive",
    estimator  = "LGBMRegressor",
)

cv_recursive, cv_explanation = assistant.create_cv(
    profile = profile_recursive,
    plan    = plan_recursive,
)

result_recursive = assistant.backtest(
    data        = df_single,
    target      = "sales",
    date_column = "date",
    cv          = cv_recursive,
    profile     = profile_recursive,
    plan        = plan_recursive,
)

print("Explanation:")
print(result_recursive.explanation)
print("\nMetrics:")
display(result_recursive.metrics)
print("\nPredictions (first rows):")
display(result_recursive.predictions.head(10))

100%|██████████| 8/8 [00:01<00:00,  4.66it/s]

Explanation:
Initial training up to 2022-09-12, expanding window, refit every fold, 14-step horizon, 8 folds. Results — mean_absolute_error: 2.3705, mean_squared_error: 9.0072, mean_absolute_scaled_error: 0.8151, mean_absolute_percentage_error: 0.0256.

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,2.370482,9.007245,0.815135,0.025587



Predictions (first rows):


,fold,pred
2022-09-13,0,90.475588
2022-09-14,0,84.455106
2022-09-15,0,82.889417
2022-09-16,0,84.671920
2022-09-17,0,88.823042
2022-09-18,0,91.621464
2022-09-19,0,93.533319
2022-09-20,0,89.746545
2022-09-21,0,84.521580
2022-09-22,0,82.889417


In [5]:
# Inspect the generated code
print(result_recursive.code)

import pandas as pd
from lightgbm import LGBMRegressor
from skforecast.preprocessing import RollingFeatures
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import TimeSeriesFold, backtesting_forecaster

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

window_features = RollingFeatures(
    stats        = ['mean', 'std', 'mean', 'mean', 'mean'],
    window_sizes = [3, 3, 7, 14, 21],
)

# Create forecaster
forecaster = ForecasterRecursive(
    estimator            = LGBMRegressor(random_state=123, verbose=-1),
    lags                 = [1, 2, 3, 4, 5, 7, 8],
    window_features      = window_features,
    categorical_features = 'auto',
    dropna_from_series   = False,
)

# Time series cross-validation configuration
cv = TimeSeriesFold(
    steps              = 14,
    initial_train_size = '2022-09-12',
    refit              = True,

### With exogenous variables

In [6]:
# cv = TimeSeriesFold(...)

result_recursive_exog = assistant.backtest(
    data        = df_single_exog,
    target      = "sales",
    date_column = "date",
    cv          = cv_recursive,
    forecaster  = "ForecasterRecursive",
    estimator   = "LGBMRegressor",
)

print("Explanation:")
print(result_recursive_exog.explanation)
print("\nMetrics:")
display(result_recursive_exog.metrics)
print("\nPredictions (first rows):")
display(result_recursive_exog.predictions.head(10))

100%|██████████| 8/8 [00:01<00:00,  5.53it/s]

Explanation:
Initial training up to 2022-09-12, expanding window, refit every fold, 14-step horizon, 8 folds. Results — mean_absolute_error: 2.4932, mean_squared_error: 9.3882, mean_absolute_scaled_error: 0.8573, mean_absolute_percentage_error: 0.0268.

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,2.49316,9.38821,0.85732,0.026843



Predictions (first rows):


,fold,pred
2022-09-13,0,90.261387
2022-09-14,0,84.638881
2022-09-15,0,83.386485
2022-09-16,0,85.394331
2022-09-17,0,88.542623
2022-09-18,0,92.004124
2022-09-19,0,94.067743
2022-09-20,0,90.447375
2022-09-21,0,85.072054
2022-09-22,0,82.567811


In [7]:
# Inspect the generated code (note exog handling)
print(result_recursive_exog.code)

import pandas as pd
from lightgbm import LGBMRegressor
from skforecast.preprocessing import RollingFeatures
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import TimeSeriesFold, backtesting_forecaster

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

window_features = RollingFeatures(
    stats        = ['mean', 'std', 'mean', 'mean', 'mean'],
    window_sizes = [3, 3, 7, 14, 21],
)

# Create forecaster
forecaster = ForecasterRecursive(
    estimator            = LGBMRegressor(random_state=123, verbose=-1),
    lags                 = [1, 2, 3, 4, 5, 7, 8],
    window_features      = window_features,
    categorical_features = 'auto',
    dropna_from_series   = False,
)

# Time series cross-validation configuration
cv = TimeSeriesFold(
    steps              = 14,
    initial_train_size = '2022-09-12',
    refit              = True,

---
## 2. ForecasterDirect (single series)

Trains one model per step. Better for long horizons or non-stationary DGPs.

### Without exogenous variables

In [8]:
profile_direct = assistant.profile(
    data        = df_single,
    target      = "sales",
    date_column = "date",
)

plan_direct = assistant.plan(
    profile    = profile_direct,
    steps      = 14,
    forecaster = "ForecasterDirect",
    estimator  = "Ridge",
)

cv_direct, _ = assistant.create_cv(
    profile = profile_direct,
    plan    = plan_direct,
)

result_direct = assistant.backtest(
    data        = df_single,
    target      = "sales",
    date_column = "date",
    cv          = cv_direct,
    profile     = profile_direct,
    plan        = plan_direct,
)

print("Explanation:")
print(result_direct.explanation)
print("\nMetrics:")
display(result_direct.metrics)
print("\nPredictions (first rows):")
display(result_direct.predictions.head(10))

100%|██████████| 8/8 [00:00<00:00, 230.71it/s]

Explanation:
Initial training up to 2022-09-12, expanding window, refit every fold, 14-step horizon, 8 folds. Results — mean_absolute_error: 1.8911, mean_squared_error: 6.0988, mean_absolute_scaled_error: 0.6503, mean_absolute_percentage_error: 0.0204.

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,1.891054,6.098757,0.650275,0.02041



Predictions (first rows):


,fold,pred
2022-09-13,0,90.550061
2022-09-14,0,86.101003
2022-09-15,0,83.187772
2022-09-16,0,83.918989
2022-09-17,0,87.729532
2022-09-18,0,91.534834
2022-09-19,0,92.769524
2022-09-20,0,90.512839
2022-09-21,0,86.172097
2022-09-22,0,83.361049


In [9]:
print(result_direct.code)

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from skforecast.preprocessing import RollingFeatures
from skforecast.direct import ForecasterDirect
from skforecast.model_selection import TimeSeriesFold, backtesting_forecaster

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

window_features = RollingFeatures(
    stats        = ['mean', 'std', 'mean', 'mean', 'mean'],
    window_sizes = [3, 3, 7, 14, 21],
)

# Create forecaster
forecaster = ForecasterDirect(
    estimator            = Ridge(),
    steps                = 14,
    lags                 = [1, 2, 3, 4, 5, 7, 8],
    window_features      = window_features,
    transformer_y        = StandardScaler(),
    categorical_features = 'auto',
    dropna_from_series   = False,
)

# Time series cross-validation configuration
cv = TimeSeriesFold(
    steps     

### With exogenous variables

In [10]:
result_direct_exog = assistant.backtest(
    data        = df_single_exog,
    target      = "sales",
    date_column = "date",
    cv          = cv_direct,
    forecaster  = "ForecasterDirect",
    estimator   = "Ridge",
)

print("Explanation:")
print(result_direct_exog.explanation)
print("\nMetrics:")
display(result_direct_exog.metrics)
print("\nPredictions (first rows):")
display(result_direct_exog.predictions.head(10))

100%|██████████| 8/8 [00:00<00:00, 201.60it/s]

Explanation:
Initial training up to 2022-09-12, expanding window, refit every fold, 14-step horizon, 8 folds. Results — mean_absolute_error: 1.9099, mean_squared_error: 6.2513, mean_absolute_scaled_error: 0.6568, mean_absolute_percentage_error: 0.0206.

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,1.909896,6.251316,0.656754,0.020578



Predictions (first rows):


,fold,pred
2022-09-13,0,90.499065
2022-09-14,0,85.915471
2022-09-15,0,83.012958
2022-09-16,0,83.528761
2022-09-17,0,87.479289
2022-09-18,0,91.188263
2022-09-19,0,92.395066
2022-09-20,0,90.411094
2022-09-21,0,85.757542
2022-09-22,0,82.628759


In [11]:
print(result_direct_exog.code)

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from skforecast.preprocessing import RollingFeatures
from skforecast.direct import ForecasterDirect
from skforecast.model_selection import TimeSeriesFold, backtesting_forecaster

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

window_features = RollingFeatures(
    stats        = ['mean', 'std', 'mean', 'mean', 'mean'],
    window_sizes = [3, 3, 7, 14, 21],
)

transformer_exog = StandardScaler()

# Create forecaster
forecaster = ForecasterDirect(
    estimator            = Ridge(),
    steps                = 14,
    lags                 = [1, 2, 3, 4, 5, 7, 8],
    window_features      = window_features,
    transformer_y        = StandardScaler(),
    transformer_exog     = transformer_exog,
    categorical_features = 'auto',
    dropna_from_series   = False,
)

---
## 3. ForecasterRecursiveMultiSeries (multi-series)

Global model trained on all series simultaneously. Ideal when series share common patterns.

### Without exogenous variables

In [12]:
profile_multi = assistant.profile(
    data             = df_multi,
    target           = "revenue",
    date_column      = "date",
    series_id_column = "series_id",
)

plan_multi = assistant.plan(
    profile    = profile_multi,
    steps      = 7,
    forecaster = "ForecasterRecursiveMultiSeries",
    estimator  = "LGBMRegressor",
)

cv_multi, _ = assistant.create_cv(
    profile = profile_multi,
    plan    = plan_multi,
)

result_multi = assistant.backtest(
    data             = df_multi,
    target           = "revenue",
    date_column      = "date",
    series_id_column = "series_id",
    cv               = cv_multi,
    profile          = profile_multi,
    plan             = plan_multi,
)

print("Explanation:")
print(result_multi.explanation)
print("\nMetrics (per series):")
display(result_multi.metrics)
print("\nPredictions (first rows):")
display(result_multi.predictions.head(10))

100%|██████████| 9/9 [00:02<00:00,  3.51it/s]

Explanation:
Initial training up to 2022-05-20, expanding window, refit every fold, 7-step horizon, 9 folds. Results — mean_absolute_error: 2.0599, mean_squared_error: 7.0944, mean_absolute_scaled_error: 2.2594, mean_absolute_percentage_error: 0.0145.

Metrics (per series):


,levels,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,store_A,1.667983,3.574361,2.043397,0.017062
1,store_B,3.018746,14.630144,2.400111,0.014794
2,store_C,1.493003,3.078723,2.325298,0.011584
3,average,2.059911,7.094409,2.256269,0.014480
4,weighted_average,2.059911,7.094409,2.256269,0.014480
5,pooling,2.059911,7.094409,2.275221,0.014480



Predictions (first rows):


,level,fold,pred
2022-05-21,store_A,0,94.087096
2022-05-21,store_B,0,203.139710
2022-05-21,store_C,0,133.217790
2022-05-22,store_A,0,94.087096
2022-05-22,store_B,0,203.139710
2022-05-22,store_C,0,133.217790
2022-05-23,store_A,0,94.272179
2022-05-23,store_B,0,203.139710
2022-05-23,store_C,0,133.217790
2022-05-24,store_A,0,94.347637


In [13]:
print(result_multi.code)

import pandas as pd
from lightgbm import LGBMRegressor
from skforecast.preprocessing import RollingFeatures, reshape_series_long_to_dict
from skforecast.recursive import ForecasterRecursiveMultiSeries
from skforecast.model_selection import TimeSeriesFold, backtesting_forecaster_multiseries

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.sort_values('date')

# Reshape to dict format (required for backtesting multi-series)
series_dict = reshape_series_long_to_dict(
    data      = data,
    series_id = 'series_id',
    index     = 'date',
    values    = 'revenue',
    freq      = 'D',
)

window_features = RollingFeatures(
    stats        = ['mean', 'std', 'mean', 'mean'],
    window_sizes = [7, 7, 14, 21],
)

# Create forecaster
forecaster = ForecasterRecursiveMultiSeries(
    estimator            = LGBMRegressor(random_state=123, verbose=-1),
    lags                 = [1, 2, 3, 4, 5, 7],
    window_features      = window_features,


### With exogenous variables

In [14]:
result_multi_exog = assistant.backtest(
    data             = df_multi_exog,
    target           = "revenue",
    date_column      = "date",
    series_id_column = "series_id",
    cv               = cv_multi,
    forecaster       = "ForecasterRecursiveMultiSeries",
    estimator        = "LGBMRegressor",
)

print("Explanation:")
print(result_multi_exog.explanation)
print("\nMetrics (per series):")
display(result_multi_exog.metrics)
print("\nPredictions (first rows):")
display(result_multi_exog.predictions.head(10))

100%|██████████| 9/9 [00:02<00:00,  3.60it/s]

Explanation:
Initial training up to 2022-05-20, expanding window, refit every fold, 7-step horizon, 9 folds. Results — mean_absolute_error: 1.9679, mean_squared_error: 6.8066, mean_absolute_scaled_error: 2.1512, mean_absolute_percentage_error: 0.0138.

Metrics (per series):


,levels,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,store_A,1.607006,3.321194,1.968696,0.016434
1,store_B,2.912533,14.317709,2.315665,0.014273
2,store_C,1.384076,2.780970,2.155647,0.010741
3,average,1.967872,6.806624,2.146669,0.013816
4,weighted_average,1.967872,6.806624,2.146669,0.013816
5,pooling,1.967872,6.806624,2.173562,0.013816



Predictions (first rows):


,level,fold,pred
2022-05-21,store_A,0,93.997902
2022-05-21,store_B,0,203.660407
2022-05-21,store_C,0,132.973738
2022-05-22,store_A,0,94.011962
2022-05-22,store_B,0,203.284371
2022-05-22,store_C,0,132.983055
2022-05-23,store_A,0,94.548217
2022-05-23,store_B,0,203.686961
2022-05-23,store_C,0,132.956938
2022-05-24,store_A,0,94.338094


In [15]:
print(result_multi_exog.code)

import pandas as pd
from lightgbm import LGBMRegressor
from skforecast.preprocessing import RollingFeatures, reshape_series_long_to_dict, reshape_exog_long_to_dict
from skforecast.recursive import ForecasterRecursiveMultiSeries
from skforecast.model_selection import TimeSeriesFold, backtesting_forecaster_multiseries

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.sort_values('date')

# Reshape to dict format (required for backtesting multi-series)
series_dict = reshape_series_long_to_dict(
    data      = data,
    series_id = 'series_id',
    index     = 'date',
    values    = 'revenue',
    freq      = 'D',
)

exog_dict = reshape_exog_long_to_dict(
    data      = data[['series_id', 'date', 'promo']],
    series_id = 'series_id',
    index     = 'date',
    freq      = 'D',
)

window_features = RollingFeatures(
    stats        = ['mean', 'std', 'mean', 'mean'],
    window_sizes = [7, 7, 14, 21],
)

# Create forecaster
forecaster 

---
## 4. ForecasterDirectMultiVariate (multivariate)

Uses all series as features to predict one target series. Direct strategy.

Note: This forecaster uses all columns in `target` as both inputs and outputs — exogenous handling is implicit in the multivariate design.

### Without exogenous variables

In [16]:
profile_multivariate = assistant.profile(
    data        = df_wide,
    target      = ["temperature", "humidity", "pressure"],
    date_column = "date",
)

plan_multivariate = assistant.plan(
    profile    = profile_multivariate,
    steps      = 7,
    forecaster = "ForecasterDirectMultiVariate",
    estimator  = "Ridge",
)

cv_multivariate, _ = assistant.create_cv(
    profile = profile_multivariate,
    plan    = plan_multivariate,
)

result_multivariate = assistant.backtest(
    data        = df_wide,
    target      = ["temperature", "humidity", "pressure"],
    date_column = "date",
    cv          = cv_multivariate,
    profile     = profile_multivariate,
    plan        = plan_multivariate,
)

print("Explanation:")
print(result_multivariate.explanation)
print("\nMetrics (per series):")
display(result_multivariate.metrics)
print("\nPredictions (first rows):")
display(result_multivariate.predictions.head(10))

100%|██████████| 9/9 [00:00<00:00, 290.88it/s]

Explanation:
Initial training up to 2022-05-20, expanding window, refit every fold, 7-step horizon, 9 folds. Results — mean_absolute_error: 1.1714, mean_squared_error: 2.2489, mean_absolute_scaled_error: 2.7544, mean_absolute_percentage_error: 0.0589.

Metrics (per series):


,levels,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,temperature,1.171411,2.248896,2.754378,0.058863



Predictions (first rows):


,level,fold,pred
2022-05-21,temperature,0,20.284268
2022-05-22,temperature,0,20.002146
2022-05-23,temperature,0,19.480820
2022-05-24,temperature,0,19.270934
2022-05-25,temperature,0,18.858181
2022-05-26,temperature,0,18.571165
2022-05-27,temperature,0,18.365348
2022-05-28,temperature,1,20.189212
2022-05-29,temperature,1,19.785787
2022-05-30,temperature,1,19.294754


In [17]:
print(result_multivariate.code)

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from skforecast.preprocessing import RollingFeatures
from skforecast.direct import ForecasterDirectMultiVariate
from skforecast.model_selection import TimeSeriesFold, backtesting_forecaster_multiseries

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

window_features = RollingFeatures(
    stats        = ['mean', 'std', 'mean', 'mean'],
    window_sizes = [7, 7, 14, 21],
)

# Create forecaster
forecaster = ForecasterDirectMultiVariate(
    estimator          = Ridge(),
    level              = 'temperature',
    steps              = 7,
    lags               = [1, 2, 3, 4, 5, 7],
    window_features    = window_features,
    transformer_series = StandardScaler(),
    dropna_from_series = False,
)

# Time series cross-validation configuration
cv = TimeSeriesFold(


### With exogenous variables


In [18]:
df_wide_exog['date'].min(), df_wide_exog['date'].max()

(Timestamp('2022-01-01 00:00:00'), Timestamp('2022-07-19 00:00:00'))

In [19]:
cv_multivariate

TimeSeriesFold(
    initial_train_size    = 2022-05-20,
    steps                 = 7,
    fold_stride           = 7,
    window_size           = None,
    differentiation       = None,
    refit                 = True,
    fixed_train_size      = False,
    gap                   = 0,
    skip_folds            = None,
    allow_incomplete_fold = True,
    return_all_indexes    = False,
    verbose               = False,
)

In [20]:
result_multivariate = assistant.backtest(
    data        = df_wide_exog,
    target      = ["temperature", "humidity", "pressure"],
    date_column = "date",
    cv          = cv_multivariate,
    forecaster  = "ForecasterDirectMultiVariate",
    # estimator   = "Ridge",
)

print("Explanation:")
print(result_multivariate.explanation)
print("\nMetrics (per series):")
display(result_multivariate.metrics)
print("\nPredictions (first rows):")
display(result_multivariate.predictions.head(10))

100%|██████████| 9/9 [00:05<00:00,  1.65it/s]

Explanation:
Initial training up to 2022-05-20, expanding window, refit every fold, 7-step horizon, 9 folds. Results — mean_absolute_error: 1.3168, mean_squared_error: 3.0263, mean_absolute_scaled_error: 3.2108, mean_absolute_percentage_error: 0.0493.

Metrics (per series):


,levels,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,temperature,1.316768,3.026315,3.21078,0.049292



Predictions (first rows):


,level,fold,pred
2022-05-21,temperature,0,25.433943
2022-05-22,temperature,0,25.015084
2022-05-23,temperature,0,24.837768
2022-05-24,temperature,0,25.775492
2022-05-25,temperature,0,25.972408
2022-05-26,temperature,0,25.471145
2022-05-27,temperature,0,25.495111
2022-05-28,temperature,1,25.358654
2022-05-29,temperature,1,25.017983
2022-05-30,temperature,1,25.417363


In [21]:
print(result_multivariate.code)

import pandas as pd
from lightgbm import LGBMRegressor
from skforecast.preprocessing import RollingFeatures
from skforecast.direct import ForecasterDirectMultiVariate
from skforecast.model_selection import TimeSeriesFold, backtesting_forecaster_multiseries

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

exog_features = ['promo']

window_features = RollingFeatures(
    stats        = ['mean', 'std', 'mean', 'mean'],
    window_sizes = [7, 7, 14, 21],
)

# Create forecaster
forecaster = ForecasterDirectMultiVariate(
    estimator            = LGBMRegressor(random_state=123, verbose=-1),
    level                = 'temperature',
    steps                = 7,
    lags                 = [1, 2, 3, 4, 5, 7],
    window_features      = window_features,
    categorical_features = 'auto',
    dropna_from_series   = False,
)

# Time series cross-validation configuration
cv = Ti

---
## 5. ForecasterStats (ARIMA)

Statistical model. No external estimator needed. Supports native prediction intervals.

### Without exogenous variables

In [22]:
profile_stats = assistant.profile(
    data        = df_single,
    target      = "sales",
    date_column = "date",
)

plan_stats = assistant.plan(
    profile    = profile_stats,
    steps      = 14,
    forecaster = "ForecasterStats",
    interval   = [10, 90],
)

cv_stats, _ = assistant.create_cv(
    profile = profile_stats,
    plan    = plan_stats,
)

result_stats = assistant.backtest(
    data        = df_single,
    target      = "sales",
    date_column = "date",
    cv          = cv_stats,
    profile     = profile_stats,
    plan        = plan_stats,
)

print("Explanation:")
print(result_stats.explanation)
print("\nMetrics:")
display(result_stats.metrics)
print("\nPredictions with intervals (first rows):")
display(result_stats.predictions.head(10))

100%|██████████| 8/8 [00:09<00:00,  1.23s/it]

Explanation:
Initial training up to 2022-09-12, expanding window, refit every fold, 14-step horizon, 8 folds. Results — mean_absolute_error: 1.9869, mean_squared_error: 6.3830, mean_absolute_scaled_error: 0.6852, mean_absolute_percentage_error: 0.0213.

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,1.986883,6.383001,0.685211,0.02133



Predictions with intervals (first rows):


,fold,pred,lower_bound,upper_bound
2022-09-13,0,91.212669,89.912401,92.512937
2022-09-14,0,85.725797,83.779467,87.672127
2022-09-15,0,81.874398,79.515980,84.232816
2022-09-16,0,83.678542,81.045747,86.311337
2022-09-17,0,87.631625,84.809819,90.453431
2022-09-18,0,90.726901,87.771706,93.682096
2022-09-19,0,91.968411,88.917518,95.019304
2022-09-20,0,90.480083,87.214511,93.745655
2022-09-21,0,84.850853,81.394300,88.307406
2022-09-22,0,81.185698,77.584564,84.786832


In [23]:
print(result_stats.code)

import pandas as pd
from skforecast.stats import Arima
from skforecast.recursive import ForecasterStats
from skforecast.model_selection import TimeSeriesFold, backtesting_stats

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

# Create forecaster (Auto-ARIMA)
forecaster = ForecasterStats(
    estimator = Arima(order=None, seasonal_order=None, m=7),
)

# Time series cross-validation configuration
cv = TimeSeriesFold(
    steps              = 14,
    initial_train_size = '2022-09-12',
    refit              = True,
    fixed_train_size   = False,
)

# Run backtesting
metrics, predictions = backtesting_stats(
    forecaster        = forecaster,
    y                 = data['sales'],
    cv                = cv,
    metric            = ['mean_absolute_error', 'mean_squared_error', 'mean_absolute_scaled_error', 'mean_absolute_percentage_error'],
    interval          = [10, 

### With exogenous variables

In [24]:
result_stats_exog = assistant.backtest(
    data        = df_single_exog,
    target      = "sales",
    date_column = "date",
    cv          = cv_stats,
    forecaster  = "ForecasterStats",
    interval    = [10, 90],
)

print("Explanation:")
print(result_stats_exog.explanation)
print("\nMetrics:")
display(result_stats_exog.metrics)
print("\nPredictions with intervals (first rows):")
display(result_stats_exog.predictions.head(10))

100%|██████████| 8/8 [00:09<00:00,  1.13s/it]

Explanation:
Initial training up to 2022-09-12, expanding window, refit every fold, 14-step horizon, 8 folds. Results — mean_absolute_error: 2.6827, mean_squared_error: 13.7065, mean_absolute_scaled_error: 0.9252, mean_absolute_percentage_error: 0.0287.

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,2.682687,13.706543,0.925171,0.028696



Predictions with intervals (first rows):


,fold,pred,lower_bound,upper_bound
2022-09-13,0,89.616790,88.223232,91.010348
2022-09-14,0,84.372231,82.011129,86.733333
2022-09-15,0,81.831669,78.823313,84.840025
2022-09-16,0,84.314566,81.115346,87.513787
2022-09-17,0,88.627748,85.408359,91.847136
2022-09-18,0,91.380037,88.160649,94.599426
2022-09-19,0,91.715577,88.486479,94.944676
2022-09-20,0,89.252317,85.794630,92.710004
2022-09-21,0,84.943654,81.053395,88.833913
2022-09-22,0,82.972947,78.741521,87.204374


In [25]:
print(result_stats_exog.code)

import pandas as pd
from skforecast.stats import Arima
from skforecast.recursive import ForecasterStats
from skforecast.model_selection import TimeSeriesFold, backtesting_stats

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

# Create forecaster (Auto-ARIMA)
forecaster = ForecasterStats(
    estimator = Arima(order=None, seasonal_order=None, m=7),
)

# Time series cross-validation configuration
cv = TimeSeriesFold(
    steps              = 14,
    initial_train_size = '2022-09-12',
    refit              = True,
    fixed_train_size   = False,
)

# Run backtesting
exog_features = ['promo', 'temperature']

metrics, predictions = backtesting_stats(
    forecaster        = forecaster,
    y                 = data['sales'],
    exog              = data[exog_features],
    cv                = cv,
    metric            = ['mean_absolute_error', 'mean_squared_error', 'mean_a

---
## 6. ForecasterFoundation

In [27]:
result_found_exog = assistant.backtest(
    data        = df_single_exog,
    target      = "sales",
    date_column = "date",
    cv          = cv_stats,
    forecaster  = "ForecasterFoundation",
)

print("Explanation:")
print(result_found_exog.explanation)
print("\nMetrics:")
display(result_found_exog.metrics)
print("\nPredictions with intervals (first rows):")
display(result_found_exog.predictions.head(10))

100%|██████████| 8/8 [00:06<00:00,  1.25it/s]

Explanation:
Initial training up to 2022-09-12, expanding window, refit every fold, 14-step horizon, 8 folds. Results — mean_absolute_error: 1.7197, mean_squared_error: 4.1892, mean_absolute_scaled_error: 0.5931, mean_absolute_percentage_error: 0.0185.

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,1.719663,4.189222,0.593056,0.018518



Predictions with intervals (first rows):


,level,fold,pred
2022-09-13,sales,0,90.568848
2022-09-14,sales,0,86.075111
2022-09-15,sales,0,83.383568
2022-09-16,sales,0,84.159660
2022-09-17,sales,0,88.344742
2022-09-18,sales,0,92.067635
2022-09-19,sales,0,93.085022
2022-09-20,sales,0,90.413620
2022-09-21,sales,0,85.601433
2022-09-22,sales,0,82.706512


In [28]:
print(result_found_exog.code)

import pandas as pd
from skforecast.foundation import FoundationModel, ForecasterFoundation
from skforecast.model_selection import TimeSeriesFold, backtesting_foundation

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

exog_features = ['promo', 'temperature']

# Create foundation model (chronos-2-small)
estimator = FoundationModel(
    model_id       = 'autogluon/chronos-2-small',
    context_length = 8192,
)

# Create forecaster
forecaster = ForecasterFoundation(estimator=estimator)

# Time series cross-validation configuration
cv = TimeSeriesFold(
    steps              = 14,
    initial_train_size = '2022-09-12',
    refit              = True,
    fixed_train_size   = False,
)

# Run backtesting
metrics, predictions = backtesting_foundation(
    forecaster        = forecaster,
    series            = data['sales'],
    cv                = cv,
    metric            

---
## 7. Shortcut: `backtest()` without pre-computed profile/plan

The `backtest()` method can also be called directly with just data and a `TimeSeriesFold`. It will internally profile and plan before running.

In [29]:
# Create a simple CV manually
cv_simple = TimeSeriesFold(
    steps=14,
    initial_train_size=300,
    refit=False,
    fixed_train_size=False,
)

# Backtest with automatic profiling and planning
result_shortcut = assistant.backtest(
    data        = df_single_exog,
    target      = "sales",
    date_column = "date",
    cv          = cv_simple,
)

print("Auto-selected forecaster:", result_shortcut.plan.forecaster)
print("Auto-selected estimator:", result_shortcut.plan.estimator)
print("Uses exog:", result_shortcut.plan.use_exog)
print("\nExplanation:")
print(result_shortcut.explanation)
print("\nMetrics:")
display(result_shortcut.metrics)

Information of folds
--------------------
Number of observations used for initial training: 300
Number of observations used for backtesting: 65
    Number of folds: 5
    Number skipped folds: 0 
    Number of steps per fold: 14
    Number of steps to exclude between last observed data (last window) and predictions (gap): 0
    Last fold only includes 9 observations.

Fold: 0
    Training:   0 -- 299  (n=300)
    Validation: 300 -- 313  (n=14)
Fold: 1
    Training:   No training in this fold
    Validation: 314 -- 327  (n=14)
Fold: 2
    Training:   No training in this fold
    Validation: 328 -- 341  (n=14)
Fold: 3
    Training:   No training in this fold
    Validation: 342 -- 355  (n=14)
Fold: 4
    Training:   No training in this fold
    Validation: 356 -- 364  (n=9)



100%|██████████| 5/5 [00:00<00:00, 303.44it/s]

Auto-selected forecaster: ForecasterRecursive
Auto-selected estimator: LGBMRegressor
Uses exog: True

Explanation:
Using 82% of data (300 observations) for initial training, expanding window, no refit, 14-step horizon, 5 folds. Results — mean_absolute_error: 2.1832, mean_squared_error: 7.3149, mean_absolute_scaled_error: 0.7603, mean_absolute_percentage_error: 0.0228.

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,2.183228,7.314911,0.760312,0.022802


In [30]:
print(result_shortcut.code)

import pandas as pd
from lightgbm import LGBMRegressor
from skforecast.preprocessing import RollingFeatures
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import TimeSeriesFold, backtesting_forecaster

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

window_features = RollingFeatures(
    stats        = ['mean', 'std', 'mean', 'mean', 'mean'],
    window_sizes = [3, 3, 7, 14, 21],
)

# Create forecaster
forecaster = ForecasterRecursive(
    estimator            = LGBMRegressor(random_state=123, verbose=-1),
    lags                 = [1, 2, 3, 4, 5, 7, 8],
    window_features      = window_features,
    categorical_features = 'auto',
    dropna_from_series   = False,
)

# Time series cross-validation configuration
cv = TimeSeriesFold(
    steps              = 14,
    initial_train_size = 300,
    refit              = False,
)

# Ru